# Topic 01 · Hands-on: fixing vanishing gradients: rerunning a 7-layer sigmoid network and nine fixes in PyTorch

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/cd01-thuc-hanh-gradient-bien-mat-en.ipynb)

This notebook accompanies [Topic 01 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/thuc-hanh-gradient-bien-mat/), a deep-learning course originally in Vietnamese.
It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/cd01-thuc-hanh-gradient-bien-mat.ipynb)):
the code, the data and every check are the same.
The notebook is based on a hands-on PyTorch tutorial on vanishing gradients (*the tutorial* for short): an MLP with
7 hidden layers and sigmoid activations, trained on Fashion-MNIST, that learns nothing; then nine fixes. Here we rerun
**the tutorial's exact PyTorch code**, measure what the tutorial only describes in words, and check its explanation.

**What you'll do:**
1. Run the tutorial's original network and measure the gradient at each layer: layer 1 receives a gradient about two
   million times smaller than the output layer's.
2. Test the "saturated sigmoid" explanation, and find the real culprit: initial weights that are too small.
3. Sweep the standard deviation of the initialization: too small and the gradient vanishes, too large and the sigmoid
   saturates.
4. Run a few epochs of BatchNorm and of "Shortcut Back" with and without its hook, then read the results of all 15
   experiments, 100 epochs × 3 seeds.

**Background:** [Lesson 09 · Vanishing gradients & initialization](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/vanishing-gradient/)
(the product of the factors $w \cdot f'$), [Lesson 07 · Momentum, Adam & AdamW](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/bo-toi-uu/),
[Lesson 10 · BatchNorm, LayerNorm & RMSNorm](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/chuan-hoa/) and
[Lesson 13 · Skip connections & ResNet](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/resnet/).
**Run time:** about 1 minute on a CPU, including the first download of Fashion-MNIST (~30 MB).

Unlike the notebooks of the individual lessons (numpy from scratch), this topic uses **PyTorch**, because the point is
to rerun the tutorial's own code. The short training cells run live; the results of 100 epochs × 3 seeds (45 training
runs, about half an hour) are loaded from the very file the reel uses. The notebook checks that the epochs it runs
itself match the first epochs of those long runs.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

PyTorch, numpy and matplotlib (Colab has them preinstalled). We use 2 CPU threads, like a free Colab machine and like
the long runs: with a different number of threads, the floating-point additions are done in a different order, and
the last digits drift. The function `fmt()` prints numbers in the English format (a decimal point, commas between
thousands); `num()` prints numbers to 3 significant digits, and very small ones in the form $3.27 \cdot 10^{-9}$.

In [ ]:
import gzip
import hashlib
import json
import math
import os
import random
import urllib.request
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "2")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.optim as optim  # noqa: PLR0402  (the tutorial's import, as printed)
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split

torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)
device = torch.device("cpu")  # the numbers below are the CPU's; a GPU adds in a different order

BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def fmt(x, digits=2):
    """A number in the English format: 13,007 · 0.1 · −4.38 (never “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}"
    return s.replace("-", "−")


def pct(p, digits=2):
    return fmt(100 * p, digits) + "%"


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def num(x, sig=3):
    """sig significant digits: 0.243 · 0.0884 · 55.7 · 6.60·10⁻³ · 3.27·10⁻⁹"""
    if x == 0:
        return "0"
    e = math.floor(math.log10(abs(x)))
    if -2 <= e < 4:
        return fmt(x, max(0, sig - 1 - e))
    return f"{fmt(x / 10**e, sig - 1)}·10{str(e).translate(SUP)}"


def tick(v, _pos=None):
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return fmt(v, len(s.split(".")[1]) if "." in s else 0)


def log_tick(v, _pos=None):
    if 0.01 <= v <= 1000:
        return tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def fmt_axes(fig):
    """Axis tick labels in the English format; call right before plt.show()."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            step = max(1, math.ceil((k1 - k0) / 8))
            axis.set_major_locator(FixedLocator([10.0**k for k in range(k0, k1 + 1) if k % step == 0]))
            axis.set_minor_locator(NullLocator())
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))

## 2 · Data: Fashion-MNIST

Fashion-MNIST (Xiao, Rasul & Vollgraf, 2017) has the same format as MNIST: 60,000 training images and 10,000 test
images, each a 28 × 28 grayscale image, but of 10 kinds of clothing and footwear instead of digits. We download the
four original files from the dataset's repository and check their MD5 hashes (the same hashes torchvision uses).

The original tutorial loads the data with `torchvision.datasets.FashionMNIST(..., transform=transforms.ToTensor())`.
`ToTensor()` only divides each byte by 255, so we read the bytes directly and divide them by 255: the same numbers,
but much faster, because there is no PIL image to build for every sample at every epoch.

In [ ]:
URL = "https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/"
FILES = {  # the standard MD5 hashes of Fashion-MNIST
    "train-images-idx3-ubyte.gz": "8d4fb7e6c68d591d4c3dfef9ec88bf0d",
    "train-labels-idx1-ubyte.gz": "25c81989df183df01b3e8a0aad5dffbe",
    "t10k-images-idx3-ubyte.gz": "bef4ecab320f06d8554ea6380940ec79",
    "t10k-labels-idx1-ubyte.gz": "bb300cfdad3c16e7a12a480ee83cd310",
}
DATA = Path("fashion")
DATA.mkdir(exist_ok=True)
for name, md5 in FILES.items():
    p = DATA / name
    if not p.exists():
        urllib.request.urlretrieve(URL + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"


def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"
    return json.loads(p.read_text())


RUNS = fetch_json("vg-tutorial.json", "3d844c521e5ab90fb7a5c39401852ce9")

In [ ]:
CLASSES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]


def load(kind):
    """The IDX format: the first 16 bytes of an image file (8 bytes of a label file) are the header; the rest are bytes
    0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        x = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 1, 28, 28)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        y = np.frombuffer(f.read(), np.uint8, offset=8)
    return torch.from_numpy(x.astype(np.float32) / 255.0), torch.from_numpy(y.astype(np.int64))


X_all, y_all = load("train")
X_test_all, y_test_all = load("t10k")
print("training:", tuple(X_all.shape), " test:", tuple(X_test_all.shape))

fig, axes = plt.subplots(1, 10, figsize=(11, 1.6))
for c, ax in enumerate(axes):
    i = int((y_test_all == c).nonzero()[0, 0])
    ax.imshow(X_test_all[i, 0], cmap="gray_r")
    ax.set_title(CLASSES[c], fontsize=8)
    ax.axis("off")
plt.show()

Each panel is the first image of one class in the test set, drawn dark on a light background; the class name is
above it.

## 3 · The tutorial's original network

This is the tutorial's code, copied verbatim: the `set_seed` function, the 90/10 split with `random_split`, the
`DataLoader`s with `batch_size = 512` (the tutorial uses this variable without assigning it; 512 is the value in its
solution notebook), and the network 784 → 7 × 128 → 10, with a sigmoid after each hidden layer, weights initialized
from $\mathcal{N}(0,\ 0.05^2)$ and biases at 0.

There is one change: the `Dataset` is the tensors divided by 255 above (`TensorDataset`) instead of `FashionMNIST` +
`ToTensor()`. Same images, same shuffling order (still a `DataLoader`), same computation; only the step that builds
PIL images is gone.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


SEED = 42
batch_size = 512
train_dataset = TensorDataset(X_all, y_all)
test_dataset = TensorDataset(X_test_all, y_test_all)


def make_loaders(seed=SEED):
    """The tutorial: set_seed(SEED), then random_split 90/10 and three DataLoaders; test_loader is left out here,
    because the test-set results are loaded from the results file."""
    set_seed(seed)
    train_size = int(len(train_dataset) * 0.9)
    train_subset, val_subset = random_split(train_dataset, [train_size, len(train_dataset) - train_size])
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader


class MLP(nn.Module):
    def __init__(self, input_dims, hidden_dims, output_dims, std=0.05):
        super(MLP, self).__init__()  # noqa: UP008  (the tutorial's code, as printed)
        self.layer1 = nn.Linear(input_dims, hidden_dims)
        self.layer2 = nn.Linear(hidden_dims, hidden_dims)
        self.layer3 = nn.Linear(hidden_dims, hidden_dims)
        self.layer4 = nn.Linear(hidden_dims, hidden_dims)
        self.layer5 = nn.Linear(hidden_dims, hidden_dims)
        self.layer6 = nn.Linear(hidden_dims, hidden_dims)
        self.layer7 = nn.Linear(hidden_dims, hidden_dims)
        self.output = nn.Linear(hidden_dims, output_dims)

        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=std)  # tutorial: std=0.05 (section 5 tries 1.0 and 10.0)
                nn.init.constant_(module.bias, 0.0)

    def forward(self, x):
        x = nn.Flatten()(x)
        x = self.layer1(x)
        x = nn.Sigmoid()(x)
        x = self.layer2(x)
        x = nn.Sigmoid()(x)
        x = self.layer3(x)
        x = nn.Sigmoid()(x)
        x = self.layer4(x)
        x = nn.Sigmoid()(x)
        x = self.layer5(x)
        x = nn.Sigmoid()(x)
        x = self.layer6(x)
        x = nn.Sigmoid()(x)
        x = self.layer7(x)
        x = nn.Sigmoid()(x)
        out = self.output(x)
        return out

The training loop is the tutorial's (SGD, `lr = 1e-2`, `CrossEntropyLoss`), with one addition: after each
`loss.backward()`, it records the mean $|\partial L / \partial W|$ of each `Linear` layer, then averages it over
the batches of the epoch. That is exactly the quantity the tutorial plots as its "Gradient Flow" chart without
printing it.

In [ ]:
def linears(model):
    return [m for m in model.modules() if isinstance(m, nn.Linear)]


def evaluate(model, loader, criterion):
    model.eval()
    loss = correct = count = 0
    with torch.no_grad():
        for X, y in loader:
            out = model(X)
            loss += criterion(out, y).item()
            correct += (torch.argmax(out, 1) == y).sum().item()
            count += len(y)
    return loss / len(loader), correct / count


def train(model, optimizer, loaders, epochs, before_epoch=None):
    train_loader, val_loader = loaders
    criterion = nn.CrossEntropyLoss()
    hist = []
    for epoch in range(epochs):
        if before_epoch:
            before_epoch(epoch)
        model.train()
        layers = linears(model)
        g = [0.0] * len(layers)
        train_loss = 0.0
        for X_train, y_train in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(X_train), y_train)
            loss.backward()
            for k, m in enumerate(layers):
                g[k] += m.weight.grad.abs().mean().item()
            optimizer.step()
            train_loss += loss.item()
        _, val_acc = evaluate(model, val_loader, criterion)
        hist.append(
            {
                "train_loss": train_loss / len(train_loader),
                "val_acc": val_acc,
                "grad": [v / len(train_loader) for v in g],
            }
        )
        print(f"epoch {epoch + 1}: train loss {fmt(hist[-1]['train_loss'], 4)} · val {pct(val_acc)}")
    return hist


loaders = make_loaders()
model = MLP(784, 128, 10)
base = train(model, optim.SGD(model.parameters(), lr=1e-2), loaders, epochs=2)

These two epochs must match the first two epochs of the long run (100 epochs, same code, same seed 42), which we load
from the results file. On the same kind of CPU they match to the last digit; on a GPU or a different CPU, the last
digits may differ slightly.

In [ ]:
stored = RUNS["runs"]["baseline"]
for e in range(2):
    print(f"epoch {e + 1}: just run {fmt(base[e]['train_loss'], 6)} · long run {fmt(stored['trainLoss'][e], 6)}")
assert abs(base[0]["train_loss"] - 2.3083373443) < 1e-3, "differs from the reel: check the data and the thread count"
assert abs(base[0]["val_acc"] - 0.0936667) < 0.005
print(
    f"\nThe tutorial printed after 100 epochs: val {pct(RUNS['paper']['baseline']['val'])} · test {pct(RUNS['paper']['baseline']['test'])}"
)
print("Long run, 3 seeds, test:", ", ".join(pct(v) for v in stored["test"]))

After 100 epochs, the network gets 10% of the test images right, exactly as the tutorial printed: the same as guessing
at random among 10 classes. All three seeds do the same.

### 🤔 Predict first
How many times smaller is the gradient at layer 1 than the gradient at the output layer: 10 times, 1,000 times, or
more than a million times?

In [ ]:
g = base[0]["grad"]
names = [f"layer {k + 1}" for k in range(7)] + ["output layer"]
for n, v in zip(names, g):
    print(f"{n:12} {num(v)}")
ratio = g[0] / g[-1]
print(f"\nlayer 1 / output layer = {num(ratio)}: {fmt(1 / ratio / 1e6, 1)} million times smaller")
assert abs(g[0] / 3.27468624e-09 - 1) < 0.01 and abs(g[-1] / 6.60295583e-03 - 1) < 0.01

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(range(7), g[:7], color=BLUE)
ax.bar([7], g[7:], color=ORANGE, hatch="//")
ax.set_yscale("log")
ax.set_xticks(range(8), [str(k + 1) for k in range(7)] + ["out"])
ax.set(xlabel="layer (from input to output)", ylabel="mean |∂L/∂W|", title="Gradient of each layer, epoch 1")
fmt_axes(fig)
plt.show()

More than two million times. The vertical axis is a log scale: each tick is a power of ten. The orange hatched bar is
the output layer; the seven blue bars are the hidden layers, each almost an order of magnitude lower than the bar to
its right. The tutorial gives the phenomenon its right name: vanishing gradients. The early layers receive almost no
error signal, so they stand still.

## 4 · Why? Testing the "saturated sigmoid" explanation

The tutorial's explanation: the derivative of the sigmoid is always less than 0.25, so stacked over 7 layers the
product goes to 0 — "derivative saturation". We measure it directly: pass a batch of validation images through the
(untrained) network and record, at each layer,
- the mean $\sigma'(z)$, and the fraction of *saturated* neurons ($\sigma'(z) < 0.01$, that is $|z| > 4.6$);
- **the difference between images**: the standard deviation of the output across the images of the batch, averaged
  over the neurons. If it goes to 0, every image reaches the next layer as the same vector.

In [ ]:
def probe(model, X):
    """Record σ′, the saturated fraction and the difference between images at the output of each sigmoid."""
    rows = []
    h = nn.Flatten()(X)
    with torch.no_grad():
        for m in linears(model)[:-1]:
            h = torch.sigmoid(m(h))
            d = h * (1 - h)
            rows.append((d.mean().item(), (d < 0.01).float().mean().item(), h.std(0).mean().item()))
    return rows


X_probe = loaders[1].dataset[:512][0]  # the first 512 validation images, as in the long run
set_seed(SEED)
fresh = MLP(784, 128, 10)  # a new network, initialized like the original one (std 0.05), untrained
rows = probe(fresh, X_probe)
print("layer   mean σ′   saturated   difference between images")
for k, (s, sat, sp) in enumerate(rows):
    print(f"{k + 1:5}   {fmt(s, 4):>7}   {pct(sat, 0):>9}   {num(sp):>10}")

**Not a single neuron is saturated.** $\sigma'$ at all 7 layers is still around 0.24, close to its maximum of 0.25:
the sigmoid is in exactly its steepest part. What collapses is the last column: the difference between images shrinks
about 7 times at each layer, and by layer 7 every image gives almost the same vector. The reason: the weights are too
small. With $z = xW$ and 128 inputs, each layer multiplies the spread of the signal by about
$\sigma' \cdot \text{std} \cdot \sqrt{128}$; around 0 the sigmoid is almost linear, with slope $\sigma'$.

The backward pass goes through the very same matrices (transposed) and the very same $\sigma'$, so the gradient
shrinks by the same factor:

In [ ]:
slope = float(np.mean([r[0] for r in rows]))
predict = slope * 0.05 * math.sqrt(128)
hidden = [g[k] / g[k + 1] for k in range(1, 6)]  # layers 2–6, each against the next one (same size, 128 × 128)
measured = float(np.exp(np.mean(np.log(hidden))))
print(f"predicted: σ′ · std · √128 = {fmt(slope, 3)} · 0.05 · {fmt(math.sqrt(128), 2)} = {fmt(predict, 3)}")
print(f"measured: each layer multiplies the gradient by {fmt(measured, 3)} (geometric mean over layers 2 → 6)")
assert all(r[1] == 0 for r in rows) and min(r[0] for r in rows) > 0.2, "differs from the reel"
assert abs(math.log(measured / predict)) < math.log(1.5)

The two numbers agree. So the gradient vanishes here because **the small initial weights** make each layer multiply
both the signal and the gradient by about 0.14, not because the sigmoid saturates. The sigmoid's derivative ≤ 0.25 is
part of that factor, but on its own it doesn't decide the outcome: with larger weights, $\text{std} \cdot \sqrt{128}$
can make up for it (Lesson 09).

## 5 · Initialization: too small, then too large

The tutorial tries initializing with std = 1 and std = 10. We sweep a wider range, right at initialization (no
training): for each value of std, one batch of 512 images, one forward pass and one backward pass.

### 🤔 Predict first
The larger the std, the larger the gradient at layer 1. So does std 10 learn better than std 1?

In [ ]:
def at_init(std):
    set_seed(0)
    m = MLP(784, 128, 10, std=std)
    X, y = X_all[:512], y_all[:512]
    nn.CrossEntropyLoss()(m(X), y).backward()
    gr = [layer.weight.grad.abs().mean().item() for layer in linears(m)]
    return gr[0] / gr[-1], float(np.mean([r[0] for r in probe(m, X)]))


stds = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0, 3.0, 10.0]
scan = [at_init(s) for s in stds]
print("  std   layer 1 / output layer   mean σ′")
for s, (r, d) in zip(stds, scan):
    print(f"{fmt(s, 2):>5}   {num(r):>22}   {fmt(d, 3):>7}")

fig, ax1 = plt.subplots(figsize=(7, 3.4))
ax1.plot(stds, [r for r, _ in scan], "o-", color=BLUE, label="gradient, layer 1 / output layer")
ax1.axhline(1, color=GREY, lw=1, ls=":")
ax1.set(xscale="log", yscale="log", xlabel="initialization std", ylabel="layer 1 / output layer")
ax2 = ax1.twinx()
ax2.plot(stds, [d for _, d in scan], "s--", color=ORANGE, label="mean σ′")
ax2.set_ylabel("mean σ′")
ax2.set_ylim(0, 0.27)
ax2.spines["right"].set_visible(True)
fig.legend(loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.02))
fmt_axes(fig)
plt.show()

In [ ]:
for k in ("baseline", "std1", "std10"):
    r = RUNS["runs"][k]
    print(f"{k:9} test after 100 epochs, 3 seeds: {', '.join(pct(v) for v in r['test'])}")

The blue line (circles, left axis) is the gradient at layer 1 relative to the output layer; it rises with std and
crosses 1 (the dotted line) between std 1 and std 3. The orange line (squares, dashed, right axis) is the mean
$\sigma'$; it falls as std grows, because a large $z$ pushes the sigmoid out to its two flat ends. The two lines go
in opposite directions, and no std keeps both.

Std 1 learns (about 73–74%). Std 10 makes the gradient at layer 1 *larger* than at the output layer, but now the
sigmoid really is saturated: $\sigma'$ is only about 0.005, and the network is only slightly better than chance
(10–20% depending on the seed). That is the real "derivative saturation" the tutorial describes, and it is caused by
large weights, not small ones. The real way out is to choose the std according to the number of inputs (Xavier, He:
Lesson 09) or to keep the signal at the right scale throughout training (BatchNorm: next section).

## 6 · BatchNorm, and "Shortcut Back" with and without its hook

BatchNorm (Lesson 10) normalizes each layer's $z$ to mean 0 and standard deviation 1 over each batch, before the
sigmoid. The tutorial's code:

In [ ]:
class MLP_BN(nn.Module):
    def __init__(self, input_dims, hidden_dims, output_dims):
        super().__init__()
        dims = [input_dims] + [hidden_dims] * 7
        self.layers = nn.ModuleList([nn.Linear(a, hidden_dims) for a in dims[:-1]])
        self.bns = nn.ModuleList([nn.BatchNorm1d(hidden_dims) for _ in range(7)])
        self.output = nn.Linear(hidden_dims, output_dims)
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.05)
                nn.init.constant_(module.bias, 0.0)

    def forward(self, x):
        x = nn.Flatten()(x)
        for layer, bn in zip(self.layers, self.bns):
            x = nn.Sigmoid()(bn(layer(x)))
        return self.output(x)


loaders = make_loaders()
model = MLP_BN(784, 128, 10)
bn = train(model, optim.SGD(model.parameters(), lr=1e-2), loaders, epochs=1)
assert abs(bn[0]["train_loss"] - 1.9205440924) < 1e-3, "differs from the reel"

The tutorial writes BatchNorm as seven lines, `self.bn1 … self.bn7`; here they are gathered into a `ModuleList` to
keep it short. The layers are created in exactly the same order, so the initial weights are exactly the same too, and
the first epoch comes out exactly as in the long run: after just one epoch, the network already gets about 68% of the
validation images right.

**"Shortcut Back"** is the tutorial's last fix. It attaches a `ShortcutBackLayer` to each hidden layer: a *hook* that
multiplies the gradient flowing backward through that layer by $1 + \alpha/2$, with $\alpha$ increasing with depth
and decreasing over the epochs. But the docstring in the tutorial's own code says that this model also changes three
other things: ReLU instead of sigmoid, added BatchNorm, and Kaiming initialization. To find out how much the hook
contributes, we run the same model with $\alpha = 0$: the hook then multiplies by exactly 1.

In [ ]:
class ShortcutBackLayer(nn.Module):
    """A hook that multiplies the gradient flowing backward through the layer by 1 + 0.5·α (the tutorial's code, with
    shortened comments)."""

    def __init__(self, layer_idx, num_layers, alpha_init=1.0):
        super().__init__()
        self.layer_idx, self.num_layers = layer_idx, num_layers
        self.alpha = alpha_init * (layer_idx + 1) / num_layers
        self.register_buffer("grad_scale", torch.tensor(1.0))

    def forward(self, x):
        if self.training:
            x.register_hook(lambda grad: grad * (1.0 + self.alpha * 0.5))
        return x

    def update_alpha(self, epoch, max_epochs):
        a0 = 1.0 * (self.layer_idx + 1) / self.num_layers
        a1 = 0.1 * (self.layer_idx + 1) / self.num_layers
        self.alpha = a0 * (a1 / a0) ** (epoch / max(max_epochs, 1))


class ShortcutBackMLP(nn.Module):
    def __init__(self, input_dims, hidden_dims, output_dims, alpha_init=1.0):
        super().__init__()
        self.num_hidden_layers = 7
        self.input_layer = nn.Linear(input_dims, hidden_dims)
        self.hidden_layers = nn.ModuleList([nn.Linear(hidden_dims, hidden_dims) for _ in range(6)])
        self.output_layer = nn.Linear(hidden_dims, output_dims)
        self.shortcut_back_layers = nn.ModuleList([ShortcutBackLayer(i, 7, alpha_init) for i in range(7)])
        self.batch_norm_layers = nn.ModuleList([nn.BatchNorm1d(hidden_dims) for _ in range(7)])
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode="fan_in", nonlinearity="relu")
                nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        x = nn.Flatten()(x)
        layers = [self.input_layer, *self.hidden_layers]
        for i, layer in enumerate(layers):
            x = self.shortcut_back_layers[i](nn.ReLU()(self.batch_norm_layers[i](layer(x))))
        return self.output_layer(x)


def shortcut_run(hook, epochs=1):
    loaders = make_loaders()
    model = ShortcutBackMLP(784, 128, 10)
    if not hook:
        for layer in model.shortcut_back_layers:
            layer.alpha = 0.0
    before = (lambda e: [layer.update_alpha(e, 100) for layer in model.shortcut_back_layers]) if hook else None
    return train(model, optim.SGD(model.parameters(), lr=1e-2), loaders, epochs, before_epoch=before)


print("with hook:")
with_hook = shortcut_run(True)
print("without hook (α = 0):")
no_hook = shortcut_run(False)
assert abs(with_hook[0]["train_loss"] - 0.9759439719) < 1e-3 and abs(no_hook[0]["train_loss"] - 1.4839382683) < 1e-3
gain = with_hook[0]["grad"][0] / no_hook[0]["grad"][0]
print(
    f"\nepoch 1, gradient at layer 1: with hook {num(with_hook[0]['grad'][0])} · without hook "
    f"{num(no_hook[0]['grad'][0])} ({fmt(gain, 1)} times as large)"
)

At first, the hook has a clear effect: the gradient at layer 1 is more than three times as large, and the loss of the
first epoch is much lower. The tutorial says to call `update_alphas(epoch, 100)` but doesn't say when; we call it at
the start of each epoch, so the first epoch runs with the largest $\alpha$.

The tutorial's `ShortcutBackLayer` class itself also states that it does not implement the method of the paper
*Take A Shortcut Back* (Guo et al., NeurIPS 2024). That paper adds branches that connect intermediate layers to the
output during training, for spiking neural networks; here there is only the multiplication of the gradient.

### 🤔 Predict first
After 100 epochs, how many percentage points does the model with the hook gain over the model without it: about 0,
about 1, or about 10?

## 7 · All 15 experiments: 100 epochs × 3 seeds

Each experiment reruns the tutorial's exact code, with seeds 42 (the tutorial's), 43 and 44. The seed decides the
90/10 split, the initial weights and the order of the batches; the spread across the three seeds shows which
differences are real. Where the printed code doesn't run verbatim, or the tutorial doesn't give a parameter, the
changes are as small as possible:

- **Every experiment:** `batch_size = 512`, the value in the solution notebook.
- **Gradient normalization:** this layer goes after each hidden `Linear` layer, before the sigmoid (the tutorial only
  says "after the Linear layers").
- **Layer-wise training:** 15 epochs for each of the 7 stages (the tutorial doesn't say), plus one run with 50 epochs
  per stage.
- **Shortcut Back:** SGD with `lr = 1e-2` (the tutorial doesn't name the optimizer), calling `update_alphas` at the
  start of each epoch; plus the $\alpha = 0$ run above.
- **Progressive training:** the tutorial calls `get_newly_unfrozen_params` without defining it; we define it as the
  parameters of the layer just unfrozen. The optimizer only receives the parameters being trained, because
  `add_param_group` raises an error when it receives a parameter it already has. The tutorial doesn't name the
  optimizer: we use SGD with `lr = 1e-2` like every other experiment, plus one run with Adam (`lr = 1e-3`, `2e-3` for
  the newly unfrozen layer).

In [ ]:
LABELS = {
    "baseline": "original (sigmoid, std 0.05, SGD)",
    "std1": "initialization std 1",
    "std10": "initialization std 10",
    "relu": "ReLU",
    "adam": "Adam",
    "batchnorm": "BatchNorm",
    "mynorm": "hand-written normalization",
    "skip": "skip connections",
    "gradnorm": "gradient normalization",
    "layerwise": "layer-wise · 15 epochs/stage",
    "layerwise_long": "layer-wise · 50 epochs/stage",
    "progressive": "progressive · SGD",
    "progressive_adam": "progressive · Adam",
    "shortcut": "Shortcut Back (with hook)",
    "shortcut_nohook": "… without hook",
}
order = sorted(LABELS, key=lambda k: np.mean(RUNS["runs"][k]["test"]))
for k in reversed(order):
    t = RUNS["runs"][k]["test"]
    paper = f"   (the tutorial printed {pct(RUNS['paper'][k]['test'])})" if k in RUNS["paper"] else ""
    print(f"{LABELS[k]:33} {pct(min(t))} – {pct(max(t))}{paper}")
fig, ax = plt.subplots(figsize=(8, 5.4))
for i, k in enumerate(order):
    t = RUNS["runs"][k]["test"]
    ax.plot([min(t), max(t)], [i, i], color=GREY, lw=2)
    ax.scatter(
        t, [i] * 3, color=GREEN if np.mean(t) > 0.5 else ORANGE, marker="o" if np.mean(t) > 0.5 else "s", zorder=3
    )
ax.axvline(0.1, color=GREY, ls=":", lw=1)
ax.text(0.105, len(order) - 0.6, "chance", color=GREY, fontsize=8)
ax.set_yticks(range(len(order)), [LABELS[k] for k in order])
ax.set(xlabel="test-set accuracy after 100 epochs (3 seeds)", xlim=(0, 1))
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: pct(v, 0)))
plt.show()

Each row is one experiment, sorted from lowest to highest; the three points are the three seeds, joined by a grey bar.
Green circles: a mean over the three seeds above 50%; orange squares: below 50%. The dotted line is chance (10%).

- **Effective:** BatchNorm, progressive training with Adam and Shortcut Back come out on top (about 86–88%); Adam,
  hand-written normalization, ReLU and initialization with std 1 follow. The tutorial's skip connections only bridge
  two groups of layers, so they help less (about 67%).
- **Stuck at 10% when run exactly as printed:** gradient normalization (it forces the gradient at each layer to a
  standard deviation of 1 regardless of its true magnitude; the cell below shows that after the first epoch,
  $\sigma'$ at every layer is already close to 0), layer-wise training with 15 epochs per stage, and progressive
  training with SGD.
  Each stage of layer-wise training creates a new output layer from scratch; given 50 epochs per stage, it gets up to
  about 43%.
- **Progressive:** the tutorial printed 83.04% but doesn't name the optimizer. With SGD (like every other experiment)
  it gets stuck; with Adam it reaches about 88%.

In [ ]:
gn = RUNS["runs"]["gradnorm"]["act1"]
print("gradient normalization, after epoch 1: mean σ′ per layer", ", ".join(num(v, 2) for v in gn["slope"]))
print("                                       saturated fraction", ", ".join(pct(v, 0) for v in gn["saturated"]))

In [ ]:
sb, nh = RUNS["runs"]["shortcut"]["test"], RUNS["runs"]["shortcut_nohook"]["test"]
print(
    f"Shortcut Back, with hook:    {', '.join(pct(v) for v in sb)}   (the tutorial printed {pct(RUNS['paper']['shortcut']['test'])})"
)
print(f"Shortcut Back, without hook: {', '.join(pct(v) for v in nh)}")
print(
    f"mean difference: {fmt(100 * (np.mean(sb) - np.mean(nh)), 2)} points · "
    f"with hook higher in all 3 seeds: {min(sb) > max(nh)}"
)
bn_test = RUNS["runs"]["batchnorm"]["test"]
print(f"sigmoid + BatchNorm (no ReLU, no Kaiming, no hook): {', '.join(pct(v) for v in bn_test)}")

The hook adds about 0.6 points, and this time it is ahead in all three seeds. But the jump from 10% to 86% comes
almost entirely from the three things changed at the same time as it: sigmoid + BatchNorm alone already reaches about
87%. The lesson is not about any particular technique, but about how to run experiments: **change one thing at a
time**, and run several seeds before trusting a small difference.

## 8 · Try changing things

The cell below trains the original network with a different std, for 2 epochs. Try 0.05, 0.2, 0.5, 1: which std gives
the highest validation accuracy? Compared with the sweep in section 5, does that agree with "the gradient at layer 1
is large enough while the sigmoid isn't saturated yet"?

In [ ]:
STD = 0.5  # change this
loaders = make_loaders()
model = MLP(784, 128, 10, std=STD)
trial = train(model, optim.SGD(model.parameters(), lr=1e-2), loaders, epochs=2)
print(
    f"std {fmt(STD, 2)}: layer 1 / output layer at the last epoch = {num(trial[-1]['grad'][0] / trial[-1]['grad'][-1])}"
)

## 9 · Limits

- One dataset, one architecture, 100 epochs on a CPU. On a GPU, the last digits will differ (the additions are done in
  a different order), but the conclusions don't change.
- The tutorial doesn't give a few parameters (the number of epochs per stage in layer-wise training, the optimizer of
  Shortcut Back and of progressive training); our choices are listed in section 7, and a different choice could give
  different numbers.
- 3 seeds are enough to see large differences; for differences under 1 point, run more.

## Summary
- The original network (sigmoid, std 0.05, SGD) gets 10% right after 100 epochs in all 3 seeds, as the tutorial
  printed; the gradient at layer 1 is about two million times smaller than at the output layer.
- The culprit is small weights, not saturation: $\sigma' \approx 0.24$ at every layer, and each layer multiplies the
  gradient by about $\sigma' \cdot 0.05 \cdot \sqrt{128} \approx 0.14$.
- A large std fixes the gradient but saturates the sigmoid: std 10 is almost at chance level.
- BatchNorm, ReLU and Adam fix it; gradient normalization, and layer-wise and progressive training with SGD, run
  exactly as printed, are still stuck at 10%.
- "Shortcut Back" changes four things at once; the hook alone adds only about 0.6 points.

## Review questions
1. Why does the gradient still vanish when $\sigma' \approx 0.24$ at every layer?
2. What is the per-layer factor $\sigma' \cdot \text{std} \cdot \sqrt{n}$ with std 1? Why does the std 1 network still
   learn even though that factor is not 1?
3. Why does the network still not learn with std 10, even though it makes the gradient at layer 1 larger than at the
   output layer?
4. What question does the "without hook" experiment answer that the tutorial doesn't?
5. Layer-wise training creates a new `nn.Linear(128, 10)` at each stage. What effect does that have?

<details><summary>Hints and answers</summary>

1. Because the per-layer factor is also multiplied by the magnitude of the weights:
   $0.24 \cdot 0.05 \cdot \sqrt{128} \approx 0.14 < 1$, so the gradient shrinks exponentially over the 7 layers.
2. About $0.05 \cdot 1 \cdot 11.3 \approx 0.6$ ($\sigma'$ is only about 0.05 because many neurons are saturated):
   the gradient still shrinks, but much more slowly; layer 1 receives a few percent of the output layer's gradient
   (section 5) instead of a millionth.
3. With std 10, $|z|$ is very large, so $\sigma' \approx 0.005$: the output of each layer is almost only 0 or 1, and
   the loss of the first epoch is very large; the gradient is large but doesn't point in a useful direction.
4. How much of the result comes from the hook, and how much from ReLU + BatchNorm + Kaiming, changed at the same
   time.
5. Each stage starts with a random output layer, throwing away the output layer already learned; the newly added
   layers are also initialized with std 0.05, so the network runs into exactly the original problem again.
</details>

## Further reading
- H. Xiao, K. Rasul & R. Vollgraf (2017), [Fashion-MNIST](https://arxiv.org/abs/1708.07747).
- X. Glorot & Y. Bengio (2010), [Understanding the difficulty of training deep feedforward neural networks](https://proceedings.mlr.press/v9/glorot10a.html).
- K. He et al. (2015), [Delving Deep into Rectifiers](https://arxiv.org/abs/1502.01852) (Kaiming initialization).
- S. Ioffe & C. Szegedy (2015), [Batch Normalization](https://arxiv.org/abs/1502.03167).
- Y. Guo et al. (2024), [Take A Shortcut Back](https://arxiv.org/abs/2401.04486) (NeurIPS 2024).

**Back to:** [Lesson 09 · Vanishing gradients & initialization](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/vanishing-gradient/).

In [ ]:
import platform

import matplotlib

print(
    f"Python {platform.python_version()} · torch {torch.__version__} · numpy {np.__version__} · "
    f"matplotlib {matplotlib.__version__}"
)